# 03 — Frozen InceptionV3 + classifier in a single model

This notebook builds **one** Keras model that already contains the two parts:

1. **InceptionV3** pretrained on ImageNet, frozen (`trainable = False`);
2. a small **classifier head** (global average pooling → Dense 256 →
   Dropout → Softmax).

Because the backbone is frozen, a single `model.fit(...)` on the images both
computes the InceptionV3 features and trains the head on top of them. There is
no separate feature-extraction model and no `.npy` feature arrays: the two
parts stay inside the same model, and only the `fit` call decides what learns
(here: the head only, since InceptionV3 is frozen).

The notebook:

1. loads the split and the balanced training manifest;
2. builds a `tf.data` pipeline with on-the-fly augmentation;
3. builds the frozen InceptionV3 + head model;
4. compiles and trains it (the backbone does not change);
5. evaluates it on the test split;
6. saves `models/inceptionv3_transfer.keras` for notebook 04.

> **Why InceptionV3 instead of the old InceptionV1/TF-Hub model?**
> `tensorflow-hub` 0.16.1 provides a Keras 2 layer, while TensorFlow 2.21
> uses Keras 3. InceptionV3 is available directly in Keras and avoids that
> compatibility problem.

> **How to read this notebook:** every code cell is preceded by a short
> summary that explains what the step does and how to interpret its output.

## 1. Imports and paths

Imports the libraries and defines the paths of the manifests, of the saved
model and the training constants.

**How to read the output:** check the TensorFlow/Keras versions and that the
model will be written to `models/inceptionv3_transfer.keras`.

In [ ]:
from pathlib import Path
import json

import cv2
import numpy as np
import tensorflow as tf
import keras
from keras.applications import InceptionV3
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
import matplotlib.pyplot as plt

ROOT_DIR = Path.cwd()
if ROOT_DIR.name == "notebooks":
    ROOT_DIR = ROOT_DIR.parent

PROCESSED_DIR = ROOT_DIR / "data" / "processed"
MODELS_DIR = ROOT_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

SPLITS_PATH = PROCESSED_DIR / "splits.json"
BALANCED_PATH = PROCESSED_DIR / "train_balanced.json"
MODEL_PATH = MODELS_DIR / "inceptionv3_transfer.keras"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42
EPOCHS = 20
LEARNING_RATE = 1e-3
AUTOTUNE = tf.data.AUTOTUNE

print("TensorFlow:", tf.__version__)
print("Keras     :", keras.__version__)

## 2. Load the split and the balanced training manifest

Reads `splits.json` (classes, validation and test sets) and
`train_balanced.json` (oversampled training paths) created by notebooks 01
and 02, then converts every relative path into an absolute path.

**How to read the output:** `Balanced train` should be larger than the
original training size, while validation and test keep their original sizes.

In [ ]:
for path in (SPLITS_PATH, BALANCED_PATH):
    if not path.exists():
        raise FileNotFoundError(
            f"{path} not found. Run notebooks 01 and 02 first."
        )

with open(SPLITS_PATH) as f:
    splits = json.load(f)
with open(BALANCED_PATH) as f:
    balanced = json.load(f)

classes = splits["classes"]
num_classes = len(classes)
id_to_class = {int(k): v for k, v in splits["id_to_class"].items()}
IMG_SIZE = tuple(splits.get("img_size", IMG_SIZE))

def absolute(path: str) -> str:
    return str((ROOT_DIR / path).resolve())

train_paths = [absolute(p) for p in balanced["paths"]]
train_labels = np.asarray(balanced["labels"], dtype=np.int32)
val_paths = [absolute(p) for p in splits["val"]["paths"]]
val_labels = np.asarray(splits["val"]["labels"], dtype=np.int32)
test_paths = [absolute(p) for p in splits["test"]["paths"]]
test_labels = np.asarray(splits["test"]["labels"], dtype=np.int32)

print("Classes       :", classes)
print("Balanced train:", len(train_paths))
print("Validation    :", len(val_paths))
print("Test          :", len(test_paths))

## 3. Build the `tf.data` pipeline

Defines the image loader (RGB, 224x224, pixels in `[0, 255]`) and a light
augmentation (horizontal flip + Gaussian noise) applied in memory. The
training set is shuffled because the model now sees images, not pre-computed
features.

**How to read the output:** one batch of shape `(32, 224, 224, 3)`. Nothing is
written to disk: augmentation happens while the batch is built.

In [ ]:
def load_image(path, label):
    def _load(path_tensor):
        if isinstance(path_tensor, tf.Tensor):
            path_tensor = path_tensor.numpy()
        path = path_tensor.decode("utf-8") if hasattr(path_tensor, "decode") else str(path_tensor)
        image = cv2.imread(path)
        if image is None:
            raise FileNotFoundError(f"Image not found or cannot be read: {path}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, IMG_SIZE)
        return image.astype(np.float32)

    image = tf.numpy_function(_load, [path], tf.float32)
    image.set_shape((IMG_SIZE[1], IMG_SIZE[0], 3))
    return image, label


def augment(image, label):
    # Light augmentation for the oversampled training paths.
    image = tf.image.random_flip_left_right(image)
    noise = tf.random.normal(
        shape=tf.shape(image), mean=0.0, stddev=10.0, dtype=tf.float32
    )
    image = tf.clip_by_value(image + noise, 0.0, 255.0)
    return image, label


def make_dataset(paths, labels, training=False, augment_data=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    if training:
        ds = ds.shuffle(len(paths), seed=SEED, reshuffle_each_iteration=True)

    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)

    if augment_data:
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)

    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)


train_ds = make_dataset(train_paths, train_labels, training=True, augment_data=True)
val_ds = make_dataset(val_paths, val_labels)
test_ds = make_dataset(test_paths, test_labels)

for images, labels in train_ds.take(1):
    print("Batch:", images.shape, labels.shape)

## 4. Build the model: frozen InceptionV3 + custom head

Creates a single functional model:

- `preprocess` rescales `[0, 255]` to `[-1, 1]` as InceptionV3 expects;
- `inception_v3` is the pretrained backbone with `trainable = False`;
- `global_average_pooling2d` turns its feature maps into one 2048-dim vector;
- `dense` (256, ReLU) → `dropout` → `dense_1` (softmax) is the head.

The two parts are not split into separate models: the backbone and the head
live in the same object, and only the frozen/unfrozen state of the backbone
decides what the next `fit` updates.

**How to read the output:** `model.summary()` shows the backbone layers as
parameter-free (`Trainable params: 0`) and the head as trainable. The trainable
parameter count should be a few hundred thousand, not millions.

In [ ]:
def build_model(num_classes, input_shape, dropout=0.5):
    base_model = InceptionV3(
        include_top=False,
        weights="imagenet",
        input_shape=input_shape,
        name="inception_v3",
    )
    base_model.trainable = False

    inputs = keras.Input(shape=input_shape, name="image")

    # InceptionV3 expects pixels in [-1, 1]. Rescaling is inside the
    # model so the data pipeline can keep raw [0, 255] values.
    x = keras.layers.Rescaling(1.0 / 127.5, offset=-1.0, name="preprocess")(inputs)
    x = base_model(x, training=False)
    x = keras.layers.GlobalAveragePooling2D(name="global_average_pooling2d")(x)
    x = keras.layers.Dense(256, activation="relu", name="dense")(x)
    x = keras.layers.Dropout(dropout, name="dropout")(x)
    outputs = keras.layers.Dense(num_classes, activation="softmax", name="dense_1")(x)

    return keras.Model(inputs, outputs, name="inceptionv3_classifier")


model = build_model(num_classes, IMG_SIZE + (3,))
model.summary()

## 5. Compile the model

Compiles with Adam at `1e-3` (a normal learning rate for a freshly initialised
head) and sparse categorical cross-entropy, and defines early stopping and
learning-rate reduction. Early stopping restores the best weights, so the
model saved at the end is the best one seen on validation.

**How to read the output:** no output here; the compile step only stores the
optimizer/loss configuration. A recompile is required later in notebook 04
after unfreezing the backbone.

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6,
    ),
]

## 6. Train with the backbone frozen

Trains the single model on the balanced, augmented pipeline. Gradients only
update the head: InceptionV3 acts as a fixed feature extractor inside the same
`fit` call.

**How to read the output:** `loss` should decrease and `val_accuracy` should
rise over the epochs. If `val_loss` stops improving for 5 epochs, early
stopping restores the best weights and training ends. On CPU expect several
minutes per epoch.

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

## 7. Evaluate on the test set

Runs one pass over the test set and prints the global accuracy plus the
precision, recall and F1-score of each class.

**How to read the output:** the global accuracy is one number for the whole
test set; the per-class rows show whether a specific diagnosis is missed.
A class with low recall is a class the model often fails to recognise.

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds, verbose=1)
print(f"Test loss    : {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")

test_probabilities = model.predict(test_ds, verbose=0)
test_predictions = np.argmax(test_probabilities, axis=1)

print("Reported accuracy:", round(accuracy_score(test_labels, test_predictions), 4))
print(
    classification_report(
        test_labels,
        test_predictions,
        labels=np.arange(num_classes),
        target_names=classes,
        digits=4,
        zero_division=0,
    )
)

## 8. Learning curves

Plots training/validation accuracy and loss for every epoch.

**How to read the output:** the two accuracy curves should rise together and
the two loss curves should fall together. A growing gap between training and
validation means over-fitting; completely flat curves mean the head is not
learning (learning rate too small or inputs not normalised).

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Train accuracy")
plt.plot(history.history["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Frozen InceptionV3 + head — accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Train loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Frozen InceptionV3 + head — loss")
plt.legend()

plt.tight_layout()
plt.show()

## 9. Confusion matrix

Cross-tabulates true and predicted classes on the test set.

**How to read the output:** the diagonal holds correct predictions; every
off-diagonal cell is a confusion between two diagnoses. This is more useful
than the global accuracy when classes are imbalanced.

In [ ]:
cm = confusion_matrix(
    test_labels,
    test_predictions,
    labels=np.arange(num_classes),
)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes).plot(
    xticks_rotation=45, cmap="Blues"
)
plt.title("Confusion matrix — frozen InceptionV3 + head")
plt.tight_layout()
plt.show()

## 10. Save the model

Saves the whole model (frozen backbone + trained head) to
`models/inceptionv3_transfer.keras`.

**How to read the output:** the printed path. Notebook 04 loads this exact
file, unfreezes the last InceptionV3 layers and fine-tunes it, so the head
trained here is the starting point of the next step.

In [ ]:
model.save(MODEL_PATH)
print("Model saved:", MODEL_PATH)

## Next step

Open **`04_fine_tuning.ipynb`**. It loads
`models/inceptionv3_transfer.keras`, unfreezes the last InceptionV3 layers and
fine-tunes the whole model with a small learning rate.